In [1]:
import numpy as np
from scipy.optimize import fsolve
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import pandas as pd
from scipy.interpolate import CubicSpline

# Given Values

In [121]:
#Wind Wave
Tw = 8 #s
Hw = 2.5 #m
#Design Wave
Td = 16 #s
Hd = 6 #m

#ship values
D = 8 #m
B = 30 #m
LOA = 160 #m
m = 30000 #DWT

#Dock
L = 200 #m
W = 40 #m

omega_w = 2 * np.pi / Tw  # (rad/s)
omega_d = 2 * np.pi / Td  # (rad/s)

#Assumed values
g = 9.81 #m/s^2
rho = 1025 #kg/m^3, from EngineeringToolbox

 # Equation Functions

In [108]:
def get_depth(x):
    #Function for getting depth depending on the distance from the shore
    if x <= 6:
        d = 6 * (1/50)
    if x > 6:
        d = (6 * 1/50) + ((x-6) * (1/40))
    return d

def get_cx(dock_angle):
    #Function for getting the x coordinate of the dock centroid based on the angle

    #6m contour x + 6-10m change in x
    start = (50*6) + (4 * 40)
    cx = start + ((L/2) * np.cos(dock_angle))
    return cx

#Set up the dispersion relation
def dispersion_relation(k, d, omega):
    return omega**2 - g * k * np.tanh(k * d)

def get_kr(theta_0, theta):
    # Refraction coefficient
    kr = np.sqrt(np.cos(theta_0) / np.cos(theta))
    return kr

def get_ks(k, d):
    # Shoaling coefficient
    n = (1 + (2*k*d) / np.sinh(2*k*d))
    ks = 1 / np.sqrt(n * np.tanh(k*d))
    return ks

def get_Hrs(H0, ks, kd):
    #Shoaling and diffraction eq
    return H0*ks*kd

def dock_angle_params(dock_angle, wave='wind', case='normal'):
    #Function that gets all parameters based on dock angle
    if wave == 'wind':
        #waves are perpendicular to shore
        theta0 = np.deg2rad(45)
        H0 = 2.5 #m
        T = 8 #s

    if wave == 'design':
        theta0 = np.deg2rad(-45)
        H0 = 6 #m
        T = 16 #s
        
    #angular velocity
    omega = (2 * np.pi) / T
    
    #centroid x location
    cx = get_cx(dock_angle)


    
    #minimum depth at centroid
    dc_min = get_depth(cx)
    #add in tides and sea level rise effects
    dc = dc_min + .5 * (1.5) + .5 * (3 * .001 * 50)

    if case == 'dock end':
    #use depth at dock end if doing maximum height calculations (minimum depth + 1/2 max water level)
        dc = 10 + .5 * (1.5) + .5 * (3 * .001 * 50)

    if case == 'conservative':
        #edge case for if we want to consider the absolute worst case given (minimum depth + maximum water level)
        dc = 10 + 1.5 + (3 * .001 * 50) #m

    if case == 'breakwater':
        dc_min = get_depth(750)
        dc = dc_min + .5 * (1.5) + .5 * (3 * .001 * 50)

    
    #wave number
    k_initial_guess = 0.08
    k = fsolve(dispersion_relation, k_initial_guess, args=(dc, omega))[0]
    
    #wavelength
    L = 2 * np.pi / k

    #celerity
    c = omega/k

    #diffraction angle
    theta_rad = np.arcsin(np.tanh(k*dc) * np.sin(theta0))
    theta = np.rad2deg(theta_rad)

    #shoaling coefficient
    ks = get_ks(k,dc)

    #refraction coefficent 
    kr = get_kr(theta0,theta_rad)

    #wave height
    H = H0 * ks * kr
    
    output = np.array([cx, dc, k, c, L, theta, ks, kr, H])
    return output
    


In [41]:
#Make empty datframe for output values
df = pd.DataFrame(columns=['Dock Angle', 'Centroid x (m)', 'Depth at Centroid (m)', 'k', 'c (m/s)', 'L (m)', 'theta (deg)', 'ks', 'kr', 'H (m)'])

dock_angles = np.arange(30,65,5)

#Calculate values for wind waves for a variety of angles, save output as row in dataframe
for i in range(len(dock_angles)):
    angle = np.deg2rad(dock_angles[i])
    values = dock_angle_params(angle, wave='wind')
    row = [dock_angles[i]] + values.tolist()
    df.loc[len(df)] = row

In [42]:
df['Angle Error'] = df['Dock Angle'] - df['theta (deg)']
df

,Dock Angle,Centroid x (m),Depth at Centroid (m),k,c (m/s),L (m),theta (deg),ks,kr,H (m),Angle Error
0,30.0,546.602540,14.460064,0.077730,10.104169,80.833354,34.890772,0.913862,0.928477,2.121249,-4.890772
1,35.0,541.915204,14.342880,0.077936,10.077468,80.619747,34.785253,0.914016,0.927882,2.120247,0.214747
2,40.0,536.604444,14.210111,0.078173,10.046882,80.375060,34.664546,0.914208,0.927205,2.119144,5.335454
3,45.0,530.710678,14.062767,0.078442,10.012519,80.100149,34.529138,0.914441,0.926449,2.117958,10.470862
4,50.0,524.278761,13.901969,0.078741,9.974505,79.796044,34.379607,0.914722,0.925620,2.116714,15.620393
5,55.0,517.357644,13.728941,0.079070,9.932995,79.463964,34.216625,0.915057,0.924722,2.115434,20.783375
6,60.0,510.000000,13.545000,0.079428,9.888167,79.105339,34.040968,0.915449,0.923762,2.114143,25.959032


In [43]:
#Started from 30 to 60 degrees, resulting angle varied from 31.5 to 32 degrees. Re ran from 31.5 to 32
#Make empty datframe for output values
df_test = pd.DataFrame(columns=['Dock Angle', 'Centroid x (m)', 'Depth at Centroid (m)', 'k', 'c (m/s)', 'L (m)', 'theta (deg)', 'ks', 'kr', 'H (m)'])

dock_angles = np.arange(34.7,34.8,.01)

#Calculate values for wind waves for a variety of angles, save output as row in dataframe
for i in range(len(dock_angles)):
    angle = np.deg2rad(dock_angles[i])
    values = dock_angle_params(angle, wave='wind')
    row = [dock_angles[i]] + values.tolist()
    df_test.loc[len(df_test)] = row
df_test['Angle Error %'] = ((df_test['Dock Angle'] - df_test['theta (deg)']) / df_test['theta (deg)']) * 100


df_test

,Dock Angle,Centroid x (m),Depth at Centroid (m),k,c (m/s),L (m),theta (deg),ks,kr,H (m),Angle Error %
0,34.70,542.214404,14.350360,0.077923,10.079181,80.633447,34.792017,0.914006,0.927920,2.120311,-0.264476
1,34.71,542.204467,14.350112,0.077923,10.079124,80.632992,34.791792,0.914006,0.927919,2.120308,-0.235090
2,34.72,542.194527,14.349863,0.077924,10.079067,80.632537,34.791567,0.914007,0.927917,2.120306,-0.205703
3,34.73,542.184585,14.349615,0.077924,10.079010,80.632082,34.791343,0.914007,0.927916,2.120304,-0.176316
4,34.74,542.174641,14.349366,0.077925,10.078953,80.631627,34.791118,0.914007,0.927915,2.120302,-0.146928
5,34.75,542.164694,14.349117,0.077925,10.078896,80.631172,34.790893,0.914008,0.927913,2.120300,-0.117539
6,34.76,542.154744,14.348869,0.077925,10.078840,80.630716,34.790668,0.914008,0.927912,2.120298,-0.088151
7,34.77,542.144792,14.348620,0.077926,10.078783,80.630261,34.790443,0.914008,0.927911,2.120296,-0.058761
8,34.78,542.134838,14.348371,0.077926,10.078726,80.629805,34.790218,0.914009,0.927910,2.120294,-0.029371
9,34.79,542.124880,14.348122,0.077927,10.078669,80.629349,34.789993,0.914009,0.927908,2.120292,0.000020


$\theta = 34.79^{\circ}$

In [44]:
df = pd.DataFrame(columns=['Wave Type', 'Centroid x (m)', 'd (m)', 'k', 'c (m/s)', 'L (m)', 'theta (deg)', 'ks', 'kr', 'H (m)'])

dock_angle = 34.79
dock_angle_rad = np.deg2rad(dock_angle)

wind_params =  dock_angle_params(dock_angle_rad, wave='wind')
design_params = dock_angle_params(dock_angle_rad, wave='design', case='dock end)
conservative_params = dock_angle_params(dock_angle_rad, wave='design', case='conservative')
wind_row = ['Wind'] + wind_params.tolist()
design_row = ['Design'] + design_params.tolist()
conser_row = ['Conservative'] + conservative_params.tolist()
df.loc[len(df)] = wind_row
df.loc[len(df)] = design_row
df.loc[len(df)] = conser_row

df

,Wave Type,Centroid x (m),d (m),k,c (m/s),L (m),theta (deg),ks,kr,H (m)
0,Wind,542.12488,14.348122,0.077927,10.078669,80.629349,34.789993,0.914009,0.927908,2.120292
1,Design,542.12488,10.825000,0.039223,10.012035,160.192561,-16.463170,1.149219,0.858683,5.920889
2,Conservative,542.12488,11.650000,0.037893,10.363339,165.813418,-17.058196,1.132038,0.860027,5.841496


In [45]:
xy = L * np.sin(dock_angle_rad)
L, dock_angle, xy

(200, 34.79, np.float64(114.11404828383984))

In [48]:
d_w = df['d (m)'].iloc[0]
d_d = df['d (m)'].iloc[1]

H_w = df['H (m)'].iloc[0]
H_d = df['H (m)'].iloc[1]

#values for wave theory validity chart
u_wind_x = d_w / (g * (Tw**2))
u_wind_y = H_w / (g * (Tw**2))

u_design_x = d_d / (g * (Td**2))
u_design_y = H_d / (g * (Td**2))

u_design_x, u_design_y

(np.float64(0.0043104134811416915), np.float64(0.002357642483802369))

# Trestle Values 

In [53]:
beta = np.arctan(1/40)
a = 43.8 * (1 - np.exp(-19 * np.tan(beta)))
b = 1.56 / (1 + np.exp(-19.5 * np.tan(beta)))

def get_db(H_b, T=Td):
    return H_b / (b - a * H_b / (g * T**2))

def solve_breaking_depth(H0, T, beta, theta_0=np.deg2rad(45), theta=np.deg2rad(dock_angle), tol=1e-4, max_iter=100):
    
    omega = 2*np.pi / T

    # Refraction coefficient (constant for normal incidence)
    kr = get_kr(theta_0, theta)

    # Initial guess for breaking height
    H_b = 0.8 * H0

    for i in range(max_iter):
        #get the first d_b from breaking equation
        d_b = get_db(H_b)

        #get second d_b from shoaling equations
        k_initial_guess = 0.08
        k = fsolve(dispersion_relation, k_initial_guess, args=(d_b, omega))[0]
        ks = get_ks(k, d_b)
        H_b_new = H0 * ks * kr

        #Compare values, break if solved
        if abs(H_b_new - H_b) < tol:
            return H_b_new, d_b, i+1, abs(H_b_new - H_b)  # converged

        #otherwise update H_b
        H_b = 0.5*H_b + 0.5*H_b_new

    if i == max_iter:
        print("Did not converge within max_iter")


H_b, d_b, iters, error = solve_breaking_depth(H0=6, T=16, beta=np.arctan(1/50))
print(f"Breaking height H_b = {H_b:.3f} m")
print(f"Breaking depth  d_b = {d_b:.3f} m")
print(f"Converged in {iters} iterations")
print(f"Error:  {error*100} %")

Breaking height H_b = 6.918 m
Breaking depth  d_b = 7.513 m
Converged in 12 iterations
Error:  0.006276384691350501 %


In [60]:
gamma = H_b/d_b
print('gamma_b is ' + str(round(gamma,3)))

gamma_b is 0.921


In [68]:
x_vals = np.array([460, 440, 420, 380, 360])
d_vals = np.array([get_depth(x) for x in x_vals])
h_vals = gamma * d_vals
d_mod = h_vals/0.78

#eta = 

df_a = pd.DataFrame({
    'x (m)': x_vals,
    'Depth d (m)': d_vals,
    'Wave Height h (m)': h_vals,
    'ACES vals':d_mod
})
df_a

,x (m),Depth d (m),Wave Height h (m),ACES vals
0,460,11.47,10.561968,13.540985
1,440,10.97,10.101551,12.950707
2,420,10.47,9.641134,12.360429
3,380,9.47,8.720300,11.179872
4,360,8.97,8.259883,10.589594


In [80]:
omega = (2 * np.pi) / Td


dc = get_depth(420) + 0.75 + 0.75

    #wave number
k_initial_guess = 0.08
k = fsolve(dispersion_relation, k_initial_guess, args=(dc, omega))[0]

ks = get_ks(k,dc)
kr = get_kr(np.deg2rad(-45),np.deg2rad(-16.46))
H = Hd * ks * kr
get_depth(360.4), Td, H

(8.979999999999999, 16, np.float64(5.800364767285177))

In [82]:
gamma * 8.335

np.float64(7.6751532035043715)

# Revetment Code

In [ ]:
#values from table
slopes = np.array([1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5]) 
A_vals = np.array([7.94e-3, 8.84e-3, 9.39e-3, 1.03e-2, 1.09e-2, 1.12e-2, 1.16e-2, 1.20e-2, 1.31e-2])
B_vals = np.array([20.1, 19.9, 21.6, 24.5, 28.7, 34.1, 41.0, 47.7, 55.6])

def get_Rc(A, B, T=16, H0=6, q_max=0.2):
    r = 1  # concrete revetment
    Rc = -(r * T * np.sqrt(g * H0) / B) * np.log(q_max / (A * np.sqrt(g * H0**3)))
    return Rc

def revet_db(SWL, Rc, m):
    x_toe = ((SWL + Rc) / (m - 1/50)) + 100
    return x_toe / 50

def eta_up(gamma_b, d_b):
    term1 = - (gamma_b**2 * d_b) / 16
    term2 = (1 + 8 / (3 * gamma_b**2))**(-1) * d_b
    return term1 + term2

In [107]:
#solver
def pull_vals(m, T=16, H0=6, q_max=0.2):
    idx = np.where(slopes == m)[0][0]
    A = A_vals[idx]
    B = B_vals[idx]
    Rc = get_Rc(A, B, T=T, H0=H0, q_max=q_max)

    def revet_solver(vars):
        d_b, H_b = vars
        gamma_b = H_b / d_b
        eta_n = eta_up(gamma_b, d_b)
        SWL = 3.65 + eta_n

        f1 = d_b - revet_db(SWL, Rc, m)
        f2 = H_b - gamma_b * d_b  # optional, ensures H_b = gamma_b * d_b
        return [f1, f2]

    # initial guesses for db and Hb
    x0 = [2.0, 1.5]
    sol = fsolve(revet_solver, x0)
    d_b, H_b = sol
    gamma_b = H_b / d_b
    eta_n = eta_up(gamma_b, d_b)
    return Rc, d_b, H_b, eta_n, gamma_b

#slope
m = 5
Rc, d_b, H_b, eta_n, gamma_b = pull_vals(m)
print(
    f"Rc = {Rc:.3f} "
    f"d_b = {d_b:.3f} "
    f"H_b = {H_b:.3f} "
    f"eta_n = {eta_n:.3f} "
    f"gamma_b = {gamma_b:.3f} "
)

Rc = 2.437 d_b = 2.026 H_b = 1.500 eta_n = 0.276 gamma_b = 0.741 


# Breakwater Code

In [123]:
def get_depth(x):
    #Function for getting depth depending on the distance from the shore
    if x <= 6:
        d = 6 * (1/50)
    if x > 6:
        d = (6 * 1/50) + ((x-6) * (1/40))
    return d

def get_cx(dock_angle):
    #Function for getting the x coordinate of the dock centroid based on the angle

    #6m contour x + 6-10m change in x
    start = (50*6) + (4 * 40)
    cx = start + ((L/2) * np.cos(dock_angle))
    return cx

#Set up the dispersion relation
def dispersion_relation(k, d, omega):
    return omega**2 - g * k * np.tanh(k * d)

def get_kr(theta_0, theta):
    # Refraction coefficient
    kr = np.sqrt(np.cos(theta_0) / np.cos(theta))
    return kr

def get_ks(k, d):
    # Shoaling coefficient
    n = (1 + (2*k*d) / np.sinh(2*k*d))
    ks = 1 / np.sqrt(n * np.tanh(k*d))
    return ks

def get_Hrs(H0, ks, kd):
    #Shoaling and diffraction eq
    return H0*ks*kd

def dock_angle_params(dock_angle, wave='wind', case='normal'):
    #Function that gets all parameters based on dock angle
    if wave == 'wind':
        #waves are perpendicular to shore
        theta0 = np.deg2rad(45)
        H0 = 2.5 #m
        T = 8 #s

    if wave == 'design':
        theta0 = np.deg2rad(-45)
        H0 = 6 #m
        T = 16 #s
        
    #angular velocity
    omega = (2 * np.pi) / T
    
    #centroid x location
    cx = get_cx(dock_angle)


    
    #minimum depth at centroid
    dc_min = get_depth(cx)
    #add in tides and sea level rise effects
    dc = dc_min + .5 * (1.5) + .5 * (3 * .001 * 50)

    if case == 'dock end':
    #use depth at dock end if doing maximum height calculations (minimum depth + 1/2 max water level)
        dc = 10 + .5 * (1.5) + .5 * (3 * .001 * 50)

    if case == 'conservative':
        #edge case for if we want to consider the absolute worst case given (minimum depth + maximum water level)
        dc = 10 + 1.5 + (3 * .001 * 50) #m

    if case == 'breakwater 1':
        dc_min = get_depth(699)
        dc = dc_min + .5 * (1.5) + .5 * (3 * .001 * 50)
    if case == 'breakwater 2':
        dc_min = get_depth(871)
        dc = dc_min + .5 * (1.5) + .5 * (3 * .001 * 50)

    
    #wave number
    k_initial_guess = 0.08
    k = fsolve(dispersion_relation, k_initial_guess, args=(dc, omega))[0]
    
    #wavelength
    L = 2 * np.pi / k

    #celerity
    c = omega/k

    #diffraction angle
    theta_rad = np.arcsin(np.tanh(k*dc) * np.sin(theta0))
    theta = np.rad2deg(theta_rad)

    #shoaling coefficient
    ks = get_ks(k,dc)

    #refraction coefficent 
    kr = get_kr(theta0,theta_rad)

    #wave height
    H = H0 * ks * kr
    
    output = np.array([cx, dc, k, c, L, theta, ks, kr, H])
    return output
    


In [124]:
#modify dock code for breakwater, returns the angle it should be perpendicular to 
df_test = pd.DataFrame(columns=['Dock Angle', 'Centroid x (m)', 'Depth at Centroid (m)', 'k', 'c (m/s)', 'L (m)', 'theta (deg)', 'ks', 'kr', 'H (m)'])

dock_angles = np.arange(38.6,38.7,.01)

#Calculate values for wind waves for a variety of angles, save output as row in dataframe
for i in range(len(dock_angles)):
    angle = np.deg2rad(dock_angles[i])
    values = dock_angle_params(angle, wave='wind', case='breakwater')
    row = [dock_angles[i]] + values.tolist()
    df_test.loc[len(df_test)] = row
df_test['Angle Error %'] = ((df_test['Dock Angle'] - df_test['theta (deg)']) / df_test['theta (deg)']) * 100


df_test

,Dock Angle,Centroid x (m),Depth at Centroid (m),k,c (m/s),L (m),theta (deg),ks,kr,H (m),Angle Error %
0,38.60,538.152047,14.248801,0.078104,10.055832,80.446658,34.699848,0.914150,0.927402,2.119462,11.239681
1,38.61,538.141157,14.248529,0.078104,10.055769,80.446155,34.699600,0.914150,0.927401,2.119460,11.269295
2,38.62,538.130265,14.248257,0.078105,10.055707,80.445652,34.699352,0.914151,0.927399,2.119457,11.298909
3,38.63,538.119370,14.247984,0.078105,10.055644,80.445149,34.699104,0.914151,0.927398,2.119455,11.328524
4,38.64,538.108473,14.247712,0.078106,10.055581,80.444646,34.698855,0.914152,0.927397,2.119453,11.358140
5,38.65,538.097574,14.247439,0.078106,10.055518,80.444142,34.698607,0.914152,0.927395,2.119451,11.387757
6,38.66,538.086672,14.247167,0.078107,10.055455,80.443639,34.698359,0.914152,0.927394,2.119448,11.417374
7,38.67,538.075768,14.246894,0.078107,10.055392,80.443135,34.698110,0.914153,0.927393,2.119446,11.446991
8,38.68,538.064861,14.246622,0.078108,10.055329,80.442631,34.697862,0.914153,0.927391,2.119444,11.476609
9,38.69,538.053952,14.246349,0.078108,10.055266,80.442127,34.697613,0.914154,0.927390,2.119442,11.506228


In [132]:
df1 = pd.DataFrame(columns=['Wave Type', 'Centroid x (m)', 'd (m)', 'k', 'c (m/s)', 'L (m)', 'theta (deg)', 'ks', 'kr', 'H (m)'])

dock_angle = 38.66 + 90
dock_angle_rad = np.deg2rad(dock_angle)

design_params = dock_angle_params(dock_angle_rad, wave='wind', case='breakwater 1')
design_params2 = dock_angle_params(dock_angle_rad, wave='wind', case='breakwater 2')

design_row = ['breakwater'] + design_params.tolist()
design_row2 = ['breakwater'] + design_params2.tolist()

df1.loc[len(df1)] = design_row
df1.loc[len(df1)] = design_row2


df1

,Wave Type,Centroid x (m),d (m),k,c (m/s),L (m),theta (deg),ks,kr,H (m)
0,breakwater,397.530234,18.27,0.072463,10.838615,86.708923,37.849683,0.914868,0.946308,2.164368
1,breakwater,397.530234,22.57,0.068782,11.418593,91.348743,40.272637,0.924818,0.962691,2.225784


In [134]:
dist = 150 #m
Lb = 300

#Distance to breakwater tip
x = np.sqrt(dist**2 + (L/2))
x

np.float64(150.33296378372907)

In [136]:
ks1 = df1['ks'].iloc[0]
ks2 = df1['ks'].iloc[1]

kr1 = df1['kr'].iloc[0]
kr2 = df1['kr'].iloc[1]

kd1 = .22
kd2 = .21

H = Hw*ks1*kr1*kd1 + Hw*ks2*kr2*kd2
H

np.float64(0.9435755757696365)

In [138]:
1.5 * get_depth(871)


32.6175